# PointDiT in motion: Kaggle runner

Needs a GPU accelerator, Internet on, the "PointDiT TUM RGB-D Subset" dataset attached, and a Kaggle secret `HF_TOKEN` for an account with access to `facebook/dinov3-vitl16-pretrain-lvd1689m`.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/pointdit-in-motion.git"
DINOV3_COMMIT = ""
OUTPUT_DIR = "/kaggle/working/outputs"

PROJECT_DIR = "/kaggle/working/pointdit-in-motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
DINOV3_DIR = f"{FORK_DIR}/third_party/dinov3"

In [ ]:
import os

if not os.path.isdir(PROJECT_DIR):
    !git clone -q --recurse-submodules {PROJECT_REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"

In [ ]:
if not os.path.isdir(DINOV3_DIR):
    !git clone -q https://github.com/facebookresearch/dinov3.git {DINOV3_DIR}
if DINOV3_COMMIT:
    !git -C {DINOV3_DIR} checkout -q {DINOV3_COMMIT}
else:
    print("DINOV3_COMMIT is empty; paste the hash below into the config cell to pin it.")
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

In [ ]:
!pip install -q -r {FORK_DIR}/requirements.txt -r {PROJECT_DIR}/requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python {PROJECT_DIR}/scripts/setup_weights.py

In [ ]:
import sys

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = find_sequences("/kaggle/input")
for sequence in sequences:
    print(sequence)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/smoke_test.py --sequence "{sequences[0]}" --output-dir {OUTPUT_DIR}/smoke

In [ ]:
import torch

print("torch  ", torch.__version__)
print("CUDA   ", torch.version.cuda)
print("GPU    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"